# LegalVision: verificação pública do lote ADD01–ADD06

Este notebook lê somente `index.json` e `prepared-inputs.jsonl`. Não chama rede ou modelos e não contém referências do avaliador.

Confere pares únicos, texto idêntico e ausência de anexo declarado no controle. Não contém imagens nem casos-fonte: não valida pixels, hashes contra os arquivos, composição canônica ou validade científica. Para a prática visual, use o notebook de pesquisa e a pasta neutra completa do repositório.

In [ ]:
from pathlib import Path
import json

base = Path.cwd()
index = json.loads((base / 'index.json').read_text(encoding='utf-8'))
rows = [json.loads(line) for line in (base / 'prepared-inputs.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
expected_ids = [f'ADD{i:02d}' for i in range(1, 7)]
assert index['case_ids'] == expected_ids, 'Inventário de casos divergente'
assert index['prepared_input_count'] == 12 and len(rows) == 12, 'Contagem de entradas divergente'
assert sorted({row['case_id'] for row in rows}) == expected_ids, 'IDs das entradas divergentes'
assert {row['condition'] for row in rows} == {'with_image', 'without_image'}, 'Condições divergentes'
assert all(row['status'] == 'prepared_not_executed' for row in rows), 'Status de execução divergente'
assert all('raw_response' not in row for row in rows), 'Resposta de modelo inesperada'
summary = {'cases': len(expected_ids), 'prepared_inputs': len(rows), 'inferences': 0}
summary
assert set(index) == {'case_ids', 'conditions', 'prepared_input_count', 'model_runs_performed'}, 'Campos do índice divergentes'
assert index['conditions'] == ['with_image', 'without_image'] and index['model_runs_performed'] == 0, 'Condições ou execução divergentes'
expected_pairs = {(case_id, condition) for case_id in expected_ids for condition in index['conditions']}
assert len({(r['case_id'], r['condition']) for r in rows}) == 12, 'Pares duplicados'
assert {(r['case_id'], r['condition']) for r in rows} == expected_pairs, 'Pares ausentes'
row_keys = {'record_type', 'status', 'case_id', 'condition', 'repeat', 'prompt', 'image', 'image_sha256'}
for row in rows:
    assert set(row) == row_keys and row['record_type'] == 'prepared_input', 'Campos do registro divergentes'
    assert type(row['repeat']) is int and row['repeat'] == 1, 'Repetição divergente'
    assert isinstance(row['prompt'], str) and row['prompt'].strip(), 'Prompt vazio'
    if row['condition'] == 'without_image':
        assert row['image'] is None and row['image_sha256'] is None, 'Controle com anexo'
    else:
        assert row['image'] == f"images/{row['case_id']}.png", 'Caminho do anexo divergente'
        assert isinstance(row['image_sha256'], str) and len(row['image_sha256']) == 64 and all(c in '0123456789abcdef' for c in row['image_sha256']), 'Hash declarado inválido'
for case_id in expected_ids:
    pair = [r for r in rows if r['case_id'] == case_id]
    assert pair[0]['prompt'] == pair[1]['prompt'], 'Prompts diferentes entre condições'
summary


In [ ]:
for case_id in index['case_ids']:
    conditions = sorted(row['condition'] for row in rows if row['case_id'] == case_id)
    print(case_id, conditions)
